### Basic error handling using try/except

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()
# correct
llm = init_chat_model(model="groq:qwen/qwen3.8-27b",max_tokens=200)
# incorrect
# llm = init_chat_model(model="groq:qwen/gwen3.8-27b",max_tokens=200)

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence")

chain = prompt | llm | StrOutputParser()

try:
    result = chain.invoke({"topic": "LangChain"})
    print(result)

except Exception as e:
    print(f"An error occurred: {e}")

LangChain is a modular development framework that simplifies the creation of LLM-powered applications by providing standardized interfaces for connecting large language models with external data sources, tools, and complex reasoning workflows.


Try/Except raises the error for definite errors like incorrect model names and so on which requires human-intervention to correct them.
For network issues or similar things which can be recovered automatically within some period of times or retries, we use **.with_retry** method

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model(model="groq:qwen/qwen3.8-27b",max_tokens=200)

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence")

chain = prompt | llm | StrOutputParser()

retry_mechanism = chain.with_retry(
    stop_after_attempt = 3,
    wait_exponential_jitter = True
)

response = retry_mechanism.invoke({'topic':"System"})
print(response)

A system is a set of interconnected components that function together as a unified whole to achieve a specific purpose.


### Fallbacks
Imagine your application uses a primary LLM to explain a topic. The primary model encounters a service error and retries, but still fails.
Instead of letting the entire operation fail, you can configure an alternative Runnable to handle the request.
- Retry: Try the same Runnable again.
- Fallback: If the primary Runnable fails, try another Runnable.

In [13]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model(model="groq:qwen/qwen3.8-27b",max_tokens=200)

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence")

chain = prompt | llm | StrOutputParser()

fallback_chain = RunnableLambda(lambda x : f"Sorry, I cannot explain {x['topic']} now")

safe_fallback = chain.with_fallbacks([fallback_chain])

response = safe_fallback.invoke({'topic':'Memory in LangChain'})

print(response)

In LangChain, Memory is a component that stores and retrieves previous interaction data, such as chat history, to enable conversational agents to maintain context and continuity across multiple turns.


### Multi-model Fallback

In [16]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

primary_llm = init_chat_model(model="groq:gwen/qwen3.8-27b",max_tokens=200)
fallback_llm = init_chat_model(model='groq:openai/gpt-oss-120b',max_tokens=200)

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence")

primary_chain = prompt | primary_llm | StrOutputParser() | RunnableLambda(lambda x:{"model": "qwen3.8-27b", "response": x})

fallback_chain = prompt | fallback_llm | StrOutputParser() | RunnableLambda(lambda x:{"Fallback model": "openai/gpt-oss-120b", "response": x})

safe_fallback = primary_chain.with_fallbacks([fallback_chain])

response = safe_fallback.invoke({'topic':'Memory in LangChain'})

print(response)

{'Fallback model': 'openai/gpt-oss-120b', 'response': 'Memory in LangChain is a component that persistently stores and retrieves contextual information (like past user inputs, model outputs, or external data) across turns of a conversation, enabling the chain to maintain stateful, context‑aware interactions.'}
